In [ ]:
# Installation des bibliothèques nécessaires
!pip install ultralytics gtts pydub

from ultralytics import YOLO
import cv2
from gtts import gTTS
from IPython.display import Audio, display
import os

# Chargement du modèle pré-entraîné (détecte déjà 80 classes comme voitures, personnes, chaises)
model = YOLO('yolov8n.pt')

In [ ]:
def get_spatial_position(x_center, frame_width):
    if x_center < frame_width / 3:
        return "à gauche"
    elif x_center > 2 * frame_width / 3:
        return "à droite"
    else:
        return "devant"

# Test sur une image/frame
results = model("/content/image1.png") # Remplace par ton flux caméra

for r in results:
    for box in r.boxes:
        # Extraire les coordonnées
        x1, y1, x2, y2 = box.xyxy[0]
        x_center = (x1 + x2) / 2

        # Obtenir le nom de la classe (ex: 'person')
        label = model.names[int(box.cls[0])]

        # Déterminer la position
        position = get_spatial_position(x_center, r.orig_shape[1])
        print(f"Obstacle: {label} détecté {position}")

In [ ]:
def play_warning(label_fr, position_fr):
    # Traduction simple pour l'exemple (Tu peux personnaliser pour la Darija)
    translations = {"person": "Bnadm", "car": "Tomobil", "chair": "Koursi"}
    label_darija = translations.get(label_fr, label_fr)

    pos_darija = "f'lyessar" if position_fr == "à gauche" else \
                 "f'lyemman" if position_fr == "à droite" else "goddamek"

    text = f"{label_darija} {pos_darija}"

    # Génération de l'audio
    tts = gTTS(text=text, lang='fr') # Utilise 'fr' ou une approche phonétique pour la Darija
    tts.save("warn.mp3")
    display(Audio("warn.mp3", autoplay=True))

# Exemple d'appel
play_warning("car", "devant")

partie 2


In [ ]:
!pip install gtts SpeechRecognition pydub


In [ ]:
import time
import math
from gtts import gTTS
from IPython.display import Audio, display
import speech_recognition as sr
from google.colab import files


In [ ]:
DESTINATIONS = {
    "ensias": (33.97084617376773, -6.870530384223207),
    "madinat al irfane": (33.98158045522268, -6.863130018925147),
    "marjane hay riad": (33.95664244102765, -6.848360174852717),
    "mosquee assounna": (34.01598429985946, -6.833034031234839)
}


In [ ]:
def speak(text):
    tts = gTTS(text=text, lang="fr")
    tts.save("voice.mp3")
    display(Audio("voice.mp3", autoplay=True))
    time.sleep(2)


In [ ]:
def recognize_audio():
    print("⬆️ Upload l'audio (ex: 'bghit nmchi l ensias')")
    uploaded = files.upload()
    audio_file = list(uploaded.keys())[0]

    r = sr.Recognizer()
    with sr.AudioFile(audio_file) as source:
        audio = r.record(source)

    text = r.recognize_google(audio, language="fr-FR")
    print("🗣️ Texte reconnu:", text.lower())
    return text.lower()


In [ ]:
def extract_destination(text):
    text = text.lower()

    # ENSIAS corrections
    ensias_keywords = ["ensias", "encias", "ncias", "dacia", "sia"]

    for word in ensias_keywords:
        if word in text:
            return "ensias"

    # autres destinations
    for place in DESTINATIONS.keys():
        if place in text:
            return place

    return None



In [ ]:
def get_gps_position():
    get_gps_position.lat += 0.00005
    get_gps_position.lon += 0.00005
    return get_gps_position.lat, get_gps_position.lon

get_gps_position.lat = 33.9650
get_gps_position.lon = -6.8750


In [ ]:
def distance(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dl/2)**2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1-a))


In [ ]:
text = recognize_audio()
TARGET = extract_destination(text)

if TARGET is None:
    speak("Ma fhemtch l destination")
    raise SystemExit

dest_lat, dest_lon = DESTINATIONS[TARGET]
speak(f"Navigation vers {TARGET}")

while True:
    lat, lon = get_gps_position()
    dist = distance(lat, lon, dest_lat, dest_lon)

    print(f"📏 Distance: {int(dist)} m")

    if dist > 50:
        speak("سير نيشان")
    elif dist > 15:
        speak("راك قربتي")
    elif dist > 5:
        speak("وصلتي تقريبا")
    else:
        speak("وصلتي للبلاصة")
        break

    time.sleep(3)


par KEY AIP